# Mistral-playground Demo Notebook

This notebook walks through every module in the **Mistral-playground** repo, explaining the design patterns used and letting you interact with each piece live.

| Module | What it does |
|---|---|
| `config.py` | Loads and validates all settings from `.env` |
| `logger.py` | Sets up a dual-handler logger (console + file) |
| `prompts_loader.py` | Reads prompt template files from `prompts/` |
| `llm_client.py` | Mistral API wrapper: singleton client, retry logic, tracing |
| `main.py` | Two runnable demos: basic chat and summarization |

**Tip:** Run cells top-to-bottom the first time. After that, feel free to jump to any section and experiment.

## Prerequisites

Before running anything, make sure you have:

1. **Installed dependencies:**
   ```bash
   pip install -r requirements.txt
   ```

2. **Created your `.env` file** (copy from the template):
   ```bash
   cp .env.example .env
   ```
   Then open `.env` and set your `MISTRAL_API_KEY`.

3. **Run this notebook from the repo root** so that relative imports (`config`, `llm_client`, etc.) resolve correctly.

In [ ]:
# Quick dependency check — run this first to catch missing packages early.
import importlib, sys

required = {"mistralai": "mistralai>=2.0.0", "dotenv": "python-dotenv>=1.0.0"}
missing = [pip_name for pkg, pip_name in required.items() if importlib.util.find_spec(pkg) is None]

if missing:
    print("Missing packages — run the following then restart the kernel:")
    for pkg in missing:
        print(f"  pip install {pkg}")
else:
    print("All dependencies found. Python", sys.version.split()[0])

---
## 1. Configuration — `config.py`

`config.py` is the **single source of truth** for all settings. Key design decisions:

- **`python-dotenv`** loads `.env` at import time, so secrets stay out of code.
- **Fail-fast validation**: if `MISTRAL_API_KEY` is missing, an `EnvironmentError` is raised immediately at import — no silent failures deep in the call stack.
- **Type coercion** (e.g., `int(os.getenv(...))`) happens once here, so the rest of the codebase works with typed values.
- **Sane defaults** mean the repo works out-of-the-box once you set the API key.

All other modules import from `config` instead of calling `os.getenv` themselves.

In [ ]:
import logging

try:
    import config

    key_preview = (config.MISTRAL_API_KEY[:4] + "…") if config.MISTRAL_API_KEY else "NOT SET"
    print(f"Model:            {config.MISTRAL_MODEL}")
    print(f"Max tokens:       {config.MISTRAL_MAX_TOKENS}")
    print(f"Temperature:      {config.MISTRAL_TEMPERATURE}")
    print(f"Request timeout:  {config.REQUEST_TIMEOUT}s")
    print(f"Log level:        {logging.getLevelName(config.LOG_LEVEL)}")
    print(f"Retry attempts:   {config.RETRY_MAX_ATTEMPTS}  (includes first try)")
    print(f"Retry base delay: {config.RETRY_BASE_DELAY}s")
    print(f"Retry max delay:  {config.RETRY_MAX_DELAY}s")
    print(f"API key:          {key_preview}")

except EnvironmentError as e:
    print(f"[config] {e}")
    print("Set MISTRAL_API_KEY in .env to continue.")

---
## 2. Logging — `logger.py`

`get_logger(name)` returns a standard `logging.Logger` with **two handlers**:

| Handler | Level | Destination |
|---|---|---|
| Console (`stdout`) | `LOG_LEVEL` from config (default `INFO`) | Terminal / notebook output |
| File | Always `DEBUG` | `logs/app.log` |

**Why two handlers?**
- The console shows only what matters during development (INFO+).
- The file captures everything (DEBUG) for post-mortem analysis without needing to redeploy.

**Idempotent design:** calling `get_logger("same-name")` twice returns the same logger — no duplicate log lines.

**Correlation IDs:** `llm_client.py` generates a short random `trace_id` per request and attaches it to every log line, so you can `grep` the log file for a single request's full lifecycle.

In [ ]:
try:
    from logger import get_logger, LOGS_DIR

    log = get_logger("notebook-demo")

    log.info("INFO message — visible here and in logs/app.log")
    log.warning("WARNING message — also visible here and in logs/app.log")
    log.debug("DEBUG message — only written to logs/app.log (not shown on console at INFO level)")

    print(f"\nLog file location: {LOGS_DIR / 'app.log'}")
    print("Open that file to see the DEBUG entry that is hidden from the console.")

    # Demonstrate idempotency: calling get_logger again returns the same object
    log2 = get_logger("notebook-demo")
    print(f"\nSame logger object returned on second call: {log is log2}")

except EnvironmentError as e:
    print(f"[logger] Skipped — config not loaded: {e}")

---
## 3. Prompt Templates — `prompts_loader.py`

Prompts live as plain text files in `prompts/` rather than being hardcoded strings. Benefits:

- **Separation of concerns**: change wording without touching Python code.
- **Version control**: prompt changes show up cleanly in git diffs.
- **Template variables**: use `{{PLACEHOLDER}}` syntax and replace with `.replace()` at runtime.
- **Easy to extend**: add a new file to `prompts/` and call `load_prompt("your_file.txt")`.

`load_prompt(filename)` raises `FileNotFoundError` immediately if the file is missing — no silent empty strings.

In [ ]:
# prompts_loader has no dependency on config, so it works without an API key.
from prompts_loader import load_prompt, PROMPTS_DIR
import os

print(f"Prompts directory: {PROMPTS_DIR}\n")

# List available prompt files
prompt_files = sorted(PROMPTS_DIR.glob("*.txt"))
print(f"Available prompts ({len(prompt_files)} file(s)):")
for p in prompt_files:
    print(f"  {p.name}")

print()

# Load and display both prompts
system_prompt = load_prompt("system_prompt.txt")
print(f"system_prompt.txt:\n  {system_prompt!r}\n")

summarize_template = load_prompt("summarize.txt")
print(f"summarize.txt:\n{summarize_template}")

In [ ]:
# Template variable substitution: replace {{TEXT}} with actual content at runtime.
sample_text = "Mistral AI is a French AI company known for building efficient open models."
filled_prompt = summarize_template.replace("{{TEXT}}", sample_text)

print("Filled template (ready to send to the API):")
print("-" * 50)
print(filled_prompt)

In [ ]:
# load_prompt raises FileNotFoundError immediately for missing files — no silent failures.
try:
    load_prompt("nonexistent.txt")
except FileNotFoundError as e:
    print(f"Expected FileNotFoundError: {e}")

---
## 4. LLM Client — `llm_client.py`

This module wraps the Mistral SDK with three production patterns:

### 4a. Singleton client (`get_client`)

The `Mistral` SDK object is created **once** and stored in a module-level `_client` variable. Every call to `get_client()` returns the same instance. This avoids:
- Re-initialising HTTP connection pools on every request
- Re-reading the API key repeatedly

```python
_client = None

def get_client() -> Mistral:
    global _client
    if _client is None:
        _client = Mistral(api_key=config.MISTRAL_API_KEY)
    return _client
```

In [ ]:
try:
    from llm_client import get_client

    client_a = get_client()
    client_b = get_client()

    print(f"client_a: {type(client_a).__name__} at {id(client_a):#x}")
    print(f"client_b: {type(client_b).__name__} at {id(client_b):#x}")
    print(f"Same object? {client_a is client_b}  ← singleton pattern working")

except EnvironmentError as e:
    print(f"[get_client] Skipped — config not loaded: {e}")

### 4b. The `chat()` function

`chat()` is the main public interface. It:
1. Generates a short random **trace ID** (e.g. `a3f9b12c`) for correlating log lines
2. Builds the **messages array** — system message first (if provided), then the user message
3. Logs the **request** at INFO level (model, tokens, temperature, truncated message)
4. Calls the API via the retry wrapper
5. Logs the **response** at INFO level (latency, token counts) and DEBUG (content preview)
6. Returns the assistant reply as a plain `str`

```python
chat(
    user_message: str,
    system_message: str | None = None,  # sets assistant behaviour
    model: str | None = None,           # overrides config.MISTRAL_MODEL
    max_tokens: int | None = None,      # overrides config.MISTRAL_MAX_TOKENS
    temperature: float | None = None,   # overrides config.MISTRAL_TEMPERATURE
) -> str
```

---
## 5. Basic Chat

`run_basic_chat()` in `main.py` demonstrates the simplest usage pattern:
1. Load a system prompt from a file
2. Send a hardcoded user question
3. Print the response

The system prompt shapes the assistant's personality/style for the entire conversation.

In [ ]:
try:
    from llm_client import chat
    from prompts_loader import load_prompt

    response = chat(
        user_message="Explain what Mistral AI is in two sentences.",
        system_message=load_prompt("system_prompt.txt"),
    )
    print("Assistant:", response)

except EnvironmentError as e:
    print(f"[chat] Skipped — set MISTRAL_API_KEY in .env: {e}")

In [ ]:
# run_basic_chat() in main.py is a thin wrapper around the same pattern above.
try:
    from main import run_basic_chat
    run_basic_chat()
except EnvironmentError as e:
    print(f"[run_basic_chat] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 6. Summarization — Template Pattern

`run_summarize(text)` demonstrates the **fill-template → chat** pattern:

1. Load `summarize.txt` which contains `{{TEXT}}` as a placeholder
2. Replace `{{TEXT}}` with the actual input text
3. Send the filled prompt to the API (no system message needed here)

This is the most common real-world prompt pattern — keep the structure in a file, inject data at runtime.

In [ ]:
try:
    from main import run_summarize

    run_summarize(
        "Mistral AI is a French company founded in 2023 that develops open and proprietary "
        "large language models. Their models are known for efficiency and strong performance "
        "relative to their size. They offer both open-weight models like Mistral 7B and "
        "commercial APIs with models like Mistral Large."
    )
except EnvironmentError as e:
    print(f"[run_summarize] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 7. Parameter Overrides

Every call to `chat()` can override the defaults from `config.py`. This is useful when:
- You want a **cheaper/faster model** for simple tasks (`mistral-small-latest`)
- You need **deterministic output** (set `temperature=0.0`)
- You want **creative variation** (set `temperature=1.0`)
- You need a **short reply** (set `max_tokens=50`)

### Temperature

Temperature controls randomness in the output (Mistral recommends `0.0–0.7`):

| Value | Behaviour |
|---|---|
| `0.0` | Deterministic — always picks the highest-probability token |
| `0.3–0.5` | Balanced — coherent but with some variety |
| `0.7` | Default — good for conversational responses |
| `1.0` | More creative/random (can be less coherent) |

In [ ]:
# Compare responses at different temperatures for the same prompt.
try:
    from llm_client import chat

    question = "Give me one single creative word that means 'happy'. Reply with only the word."
    print(f"Prompt: {question!r}\n")

    for temp in [0.0, 0.5, 1.0]:
        resp = chat(question, temperature=temp, max_tokens=10)
        print(f"  temperature={temp}: {resp.strip()}")

except EnvironmentError as e:
    print(f"[temperature demo] Skipped — set MISTRAL_API_KEY in .env: {e}")

In [ ]:
# Override the model and cap the response length.
# mistral-small-latest is faster and cheaper for simple factual questions.
try:
    from llm_client import chat

    resp = chat(
        user_message="What is 2 + 2? Reply with only the number.",
        model="mistral-small-latest",
        max_tokens=5,
    )
    print(f"Response (mistral-small-latest, max_tokens=5): {resp.strip()}")

except EnvironmentError as e:
    print(f"[model override demo] Skipped — set MISTRAL_API_KEY in .env: {e}")

---
## 8. Retry Logic — `_call_with_retry`

Transient API failures (rate limits, server errors) are a fact of life. `llm_client.py` implements the pattern recommended by the [Mistral API docs](https://docs.mistral.ai/api/):

### Which errors are retried?

| Status Code | Meaning | Retried? |
|---|---|---|
| `429` | Rate limit exceeded | Yes |
| `500` | Internal server error | Yes |
| `502` | Bad gateway | Yes |
| `503` | Service unavailable | Yes |
| `504` | Gateway timeout | Yes |
| `401` | Invalid API key | **No** — won't fix itself |
| `422` | Bad request payload | **No** — won't fix itself |

### Backoff formula

```
delay = min(base_delay × 2^attempt + jitter, max_delay)
```

- **Exponential** growth prevents hammering the server.
- **Jitter** (`random 0–0.5s`) prevents all clients retrying at the same instant (thundering-herd problem).
- **`Retry-After` header**: if the server sends one, use that value instead of our calculated delay.

The cell below visualises the delays for your current config.

In [ ]:
# Visualise the backoff delays — no API key needed.
import random

try:
    import config

    def simulate_backoff(seed=42):
        """Reproduce the exact delay formula from _call_with_retry."""
        random.seed(seed)
        delays = []
        for attempt in range(config.RETRY_MAX_ATTEMPTS - 1):  # last attempt has no delay
            delay = min(
                config.RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 0.5),
                config.RETRY_MAX_DELAY,
            )
            delays.append(delay)
        return delays

    delays = simulate_backoff()
    total = sum(delays)

    print(f"Config: RETRY_MAX_ATTEMPTS={config.RETRY_MAX_ATTEMPTS}, "
          f"RETRY_BASE_DELAY={config.RETRY_BASE_DELAY}s, "
          f"RETRY_MAX_DELAY={config.RETRY_MAX_DELAY}s\n")

    for i, d in enumerate(delays, 1):
        bar = "█" * int(d * 4)
        print(f"  Attempt {i} fails → wait {d:5.2f}s  {bar}")

    next_attempt = len(delays) + 1
    print(f"  Attempt {next_attempt} — final try (no more retries after this)\n")
    print(f"  Max cumulative wait before final attempt: {total:.2f}s")

except EnvironmentError as e:
    print(f"[backoff demo] Skipped — config not loaded: {e}")

In [ ]:
# Mock demonstration of retry: simulate a 429 on the first call, success on the second.
# This runs without a real API key by patching the SDK internals.
from unittest.mock import patch, MagicMock

try:
    import llm_client

    call_count = 0

    def mock_complete(**kwargs):
        global call_count
        call_count += 1
        if call_count == 1:
            exc = Exception("Status 429: Rate limit exceeded")
            exc.status_code = 429
            raise exc
        # Second call succeeds
        return MagicMock(
            choices=[MagicMock(message=MagicMock(content="Mock response after retry"))],
            usage=MagicMock(prompt_tokens=5, completion_tokens=8, total_tokens=13),
        )

    call_count = 0
    with patch.object(llm_client.get_client().chat, "complete", side_effect=mock_complete):
        result = llm_client.chat("Hello, are you there?")

    print(f"Total SDK calls made: {call_count}")
    print(f"Result: {result}")
    print("\n(Check the log output above for the WARNING about the retried 429.)")

except EnvironmentError as e:
    print(f"[mock retry demo] Skipped — config not loaded: {e}")

In [ ]:
# Non-retryable errors (like 400 Bad Request) are re-raised immediately — no delay wasted.
from unittest.mock import patch, MagicMock

try:
    import llm_client

    non_retryable_calls = 0

    def mock_400(**kwargs):
        global non_retryable_calls
        non_retryable_calls += 1
        exc = Exception("Status 400: Bad request")
        exc.status_code = 400
        raise exc

    non_retryable_calls = 0
    try:
        with patch.object(llm_client.get_client().chat, "complete", side_effect=mock_400):
            llm_client.chat("Bad payload")
    except Exception as e:
        print(f"Exception raised: {e}")
        print(f"SDK calls made: {non_retryable_calls}  (1 = raised immediately, no retry)")

except EnvironmentError as e:
    print(f"[non-retryable demo] Skipped — config not loaded: {e}")

---
## 9. Interactive Playground

Edit the cell below to experiment with your own prompts. Try changing the system message to give the assistant a different persona, or adjust the parameters to see how they affect the output.

In [ ]:
# Edit anything below and re-run the cell!
MY_SYSTEM  = "You are a pirate. Respond only in pirate speak."
MY_MESSAGE = "Tell me about the weather today."
MY_MODEL   = None          # None = use config default; e.g. "mistral-small-latest"
MY_TEMP    = None          # None = use config default; e.g. 0.0 for deterministic
MY_TOKENS  = None          # None = use config default; e.g. 100 for a short reply

try:
    from llm_client import chat
    response = chat(
        user_message=MY_MESSAGE,
        system_message=MY_SYSTEM,
        model=MY_MODEL,
        temperature=MY_TEMP,
        max_tokens=MY_TOKENS,
    )
    print(response)
except EnvironmentError as e:
    print(f"[playground] Set MISTRAL_API_KEY in .env to use this cell: {e}")

---
## 10. Next Steps

The repo is intentionally minimal — a solid foundation to extend. Here are the natural next steps described in the README:

| Feature | What to explore |
|---|---|
| **Streaming** | `client.chat.stream(...)` — print tokens as they arrive |
| **Structured output** | `response_format={"type": "json_object"}` — get JSON back reliably |
| **Function calling** | `tools=[...]` — let the model call your Python functions |
| **Async** | `AsyncMistral` + `asyncio` — parallel requests for higher throughput |
| **Observability** | Integrate with LangSmith, Weights & Biases, or OpenTelemetry |
| **Multi-turn chat** | Maintain a `messages` list and append each turn |
| **New prompts** | Add `.txt` files to `prompts/` and call `load_prompt()` |

To add a new prompt workflow:
```python
# 1. Create prompts/translate.txt:
#    Translate the following text to {{LANGUAGE}}:\n\n{{TEXT}}

# 2. Use it:
from prompts_loader import load_prompt
from llm_client import chat

template = load_prompt("translate.txt")
prompt = template.replace("{{LANGUAGE}}", "French").replace("{{TEXT}}", "Hello, world!")
print(chat(prompt))
```